![](https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true)



# DEDL - HDA Tutorial - access Energy Indicators

**Author**: EUMETSAT <br>
**Copyright**: 2024 EUMETSAT <br>
**Licence**: MIT <br>
**Modified by**: froura

<div class="alert alert-block alert-warning">
<b> Prerequisites: </b>
<li> For Data access : <a href="https://platform.destine.eu/"> DestinE user account and access to restricted datasets</a> </li>
<li> The described dataset will be available at the end of July.</a> </li>
</div>

### Import the relevant modules

In [1]:
import requests
import json
from getpass import getpass
from tqdm import tqdm
import xarray as xr

import destinelab as deauth


### Define some constants for the API URL




In [2]:
# Define the collection to be used
COLLECTION_ID = "EO.BSC.DAT.ENERGY_INDICATORS"

# Core API
HDA_API_URL = "https://hda.data.destination-earth.eu"

# STAC API
## Core
STAC_API_URL = f"{HDA_API_URL}/stac/v2"

## Collections
COLLECTIONS_URL = f"{STAC_API_URL}/collections"
COLLECTION_BY_ID_URL = f"{COLLECTIONS_URL}/{COLLECTION_ID}"

## Items
COLLECTION_ITEMS_URL = f"{COLLECTIONS_URL}/{COLLECTION_ID}/items"

## Item Search
SEARCH_URL = f"{STAC_API_URL}/search"

## HTTP Success
HTTP_SUCCESS_CODE = 200

## Authenticate

In [3]:
DESP_USERNAME = input("Please input your DESP username or email: ")
DESP_PASSWORD = getpass("Please input your DESP password: ")

auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
if access_token is not None:
    print("DEDL/DESP Access Token Obtained Successfully")
else:
    print("Failed to Obtain DEDL/DESP Access Token")

auth_headers = {"Authorization": f"Bearer {access_token}"}

Please input your DESP username or email:  francesc.roura@bsc.es
Please input your DESP password:  ········


DEDL/DESP Access Token Obtained Successfully


### Discover data - Authenticated

Once authenticated, we can discover the collection.

In [4]:
response = requests.get(COLLECTION_BY_ID_URL, headers=auth_headers)

# JSON(response.json(), expanded=False)
print(json.dumps(response.json(), indent=4))

{
    "id": "EO.BSC.DAT.ENERGY_INDICATORS",
    "description": "The Energy Indicators dataset from the Climate Adaptation Digital Twin is designed to allow the provision of energy-relevant indicators to adapt to climate change. It provides standard indicators for the wind energy sector (capacity factors for different wind turbines, weekly wind histograms, low wind and high wind events), as well as indicators for energy demand (heating and cooling degree-days). For these simultations we provide them for the north sea, at 5km resolution. The indicators have been created from houly data.",
    "stac_version": "1.1.0",
    "links": [
        {
            "rel": "http://www.opengis.net/def/rel/ogc/1.0/queryables",
            "type": "application/schema+json",
            "href": "https://hda.data.destination-earth.eu/stac/v2/collections/EO.BSC.DAT.ENERGY_INDICATORS/queryables",
            "title": "Queryables"
        },
        {
            "rel": "items",
            "type": "applicat

## Search

Once a collection is selected, you can search for items that match the specified input filters and order the results. In this example, we look at the elements of January 2010.


In [5]:
COLLECTION_ID = "EO.BSC.DAT.ENERGY_INDICATORS"
BODY = {
    "collections": [
        COLLECTION_ID,
    ],
    "datetime": "2010-01-01T00:00:00Z/2010-01-31T00:00:00Z",
    "bbox": [-9.7, 36.0, 3.5, 43.9],
    "sortby": [{"field": "datetime", "direction": "desc"}],
    "limit": 10,
}


r = requests.post(SEARCH_URL, json=BODY, headers=auth_headers)
if r.status_code != 200:
    (print(r.text))
r.raise_for_status()


print(json.dumps(r.json(), indent=4))

{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "assets": {
                "2010_01_01_T00_00_cdd.nc": {
                    "href": "https://hda-download.eumetsat.data.destination-earth.eu/data/internal_fdp/EO.BSC.DAT.ENERGY_INDICATORS/EO.BSC.DAT.ENERGY_INDICATORS_20100101T000000_20100131T230000__ifs-nemo__hist__2__1__baseline__high/2010_01_01_T00_00_cdd.nc",
                    "type": "application/netcdf",
                    "title": "2010_01_01_T00_00_cdd.nc",
                    "roles": [
                        "data"
                    ],
                    "alternate": {
                        "origin": {
                            "href": "https://s3.central.data.destination-earth.eu/usergenerated-eo.bsc.dat.energy-indicators/data/2010/01/EO.BSC.DAT.ENERGY_INDICATORS_20100101T000000_20100131T230000__ifs-nemo__hist__2__1__baseline__high/2010_01_01_T00_00_cdd.nc",
                            "type": "application/ne

## Download

Once we obtained the search results, we can download the returned data. We can download the full element corresponding to the month investigated previously (several hundreds of Gb):

In [6]:
# select the first item in the result to download
product = r.json()['features'][0]

# DownloadLink is an asset representing the whole product
download_url = product["assets"]["downloadLink"]["href"]
print("Download URL:", download_url)

ITEM_ID = product["id"]
print("Item ID:", ITEM_ID)

response = requests.get(download_url, stream=True, headers=auth_headers)

total_size = int(response.headers.get("Content-Length", 0))


# If the request was successful, download the file
if response.status_code == HTTP_SUCCESS_CODE:

    print("Downloading ...")
    filename = ITEM_ID + ".zip"

    # Show progress bar if size is known
    with open(filename, "wb") as f, tqdm(
        total=total_size, unit="B", unit_scale=True, desc="Downloading"
    ) as progress_bar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                f.flush()
                progress_bar.update(len(chunk))  # Update progress bar

else:
    print("Request Unsuccessful! Error-Code: {}".format(response.status_code))



Download URL: https://hda-download.eumetsat.data.destination-earth.eu/data/internal_fdp/EO.BSC.DAT.ENERGY_INDICATORS/EO.BSC.DAT.ENERGY_INDICATORS_20100101T000000_20100131T230000__ifs-nemo__hist__2__1__baseline__high/downloadLink
Item ID: EO.BSC.DAT.ENERGY_INDICATORS_20100101T000000_20100131T230000__ifs-nemo__hist__2__1__baseline__high


Downloading: 74.2MB [00:06, 10.8MB/s]


KeyboardInterrupt: 

## Download single files

Once we obtained the search results, we can download as well specific files.

In [12]:
url = r.json()['features'][0]['assets']['2010_01_01_T00_00_hdd.nc']['href']

local_file = "file.nc"

# download
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)


HTTPError: 403 Client Error: Forbidden for url: https://hda-download.eumetsat.data.destination-earth.eu/data/internal_fdp/EO.BSC.DAT.ENERGY_INDICATORS/EO.BSC.DAT.ENERGY_INDICATORS_20100101T000000_20100131T230000__ifs-nemo__hist__2__1__baseline__high/2010_01_01_T00_00_hdd.nc

# Display the downloaded file

The downloaded file can be accessed directly with xarray

In [13]:
ds = xr.open_dataset(local_file)

barcelona = ds.sel(lat=41.3874,lon=2.1686,method="nearest")

print(barcelona)

print(f"Latitude : {barcelona.lat.item()}")
print(f"Longitude: {barcelona.lon.item()}")
print(f"HWE: {barcelona.hwe.item()}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/froura/repositories/climatedt-community-resources/example_apps/file.nc'